# Programmatically Creating Jupyter Notebooks with Python

This notebook demonstrates how to **build an `.ipynb` file from scratch** using Python. We will cover two approaches:

1. Using the **`nbformat`** library (the official, recommended way).
2. Building the notebook as **raw JSON** with the `json` library.

By the end you'll understand the underlying structure of a notebook file (cells, metadata, and the kernel spec) and how to read it back to verify the result.

## 1. Import Required Libraries

Import the libraries we need. **`nbformat`** is the official library for reading and writing notebook files, and **`json`** lets us build a notebook as a raw dictionary for the alternative approach.

In [ ]:
import json
from pathlib import Path

import nbformat
from nbformat.v4 import new_notebook, new_markdown_cell, new_code_cell

print(f"nbformat version: {nbformat.__version__}")

## 2. Define the Notebook Structure

A notebook is essentially a container of **cells** plus some **metadata**. We start by creating an empty notebook object with `new_notebook()`. Its `cells` attribute is a list we will append to.

In [ ]:
# Create a fresh, empty notebook object
nb = new_notebook()

print(type(nb))
print(f"Notebook format: v{nb.nbformat}.{nb.nbformat_minor}")
print(f"Number of cells (initially): {len(nb.cells)}")

## 3. Add Markdown Cells

Markdown cells hold formatted documentation. Create them with `new_markdown_cell()` and append them to `nb.cells`.

In [ ]:
# Add a title and an introductory markdown cell
nb.cells.append(new_markdown_cell("# My Generated Notebook\n\nThis notebook was created entirely from Python code."))
nb.cells.append(new_markdown_cell(
    "## Overview\n\n"
    "- Built with `nbformat`\n"
    "- Contains both markdown and code cells\n"
    "- Fully runnable in Jupyter or VS Code"
))

print(f"Cells after adding markdown: {len(nb.cells)}")

## 4. Add Code Cells

Code cells contain executable Python source. Create them with `new_code_cell()`, passing the source code as a string, then append them to the notebook.

In [ ]:
# Add a couple of code cells with real Python source
nb.cells.append(new_code_cell("import numpy as np\n\narr = np.arange(10)\nprint('Array:', arr)\nprint('Sum:', arr.sum())"))
nb.cells.append(new_code_cell("squared = arr ** 2\nprint('Squared:', squared)"))

print(f"Total cells now: {len(nb.cells)}")
for i, cell in enumerate(nb.cells, start=1):
    print(f"  Cell {i}: {cell.cell_type}")

## 5. Set Notebook Metadata

The **metadata** tells Jupyter which kernel to use and what language the code is written in. Without a valid `kernelspec`, Jupyter may prompt you to pick a kernel when opening the file.

In [ ]:
# Define the kernel specification and language info
nb.metadata["kernelspec"] = {
    "display_name": "Python 3",
    "language": "python",
    "name": "python3",
}
nb.metadata["language_info"] = {
    "name": "python",
    "version": "3.10",
    "mimetype": "text/x-python",
    "file_extension": ".py",
}

print(json.dumps(nb.metadata, indent=2))

## 6. Write the Notebook to a File

Use `nbformat.write()` to serialize the notebook object to disk as a `.ipynb` file (which is really just JSON in a specific schema).

In [ ]:
# Serialize the notebook object to an .ipynb file
output_path = Path("generated_notebook.ipynb")

with output_path.open("w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print(f"Notebook written to: {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size} bytes")

## 7. Read and Verify the Notebook

Load the file back with `nbformat.read()` and inspect its structure to confirm everything was saved correctly.

In [ ]:
# Read the notebook back and verify its contents
loaded_nb = nbformat.read(output_path, as_version=4)

# Validate against the notebook schema (raises if invalid)
nbformat.validate(loaded_nb)
print("Notebook is valid!\n")

print(f"Loaded {len(loaded_nb.cells)} cells:")
for i, cell in enumerate(loaded_nb.cells, start=1):
    preview = cell.source.splitlines()[0] if cell.source else "(empty)"
    print(f"  Cell {i} [{cell.cell_type}]: {preview}")

## 8. Create a Notebook Using Raw JSON

`nbformat` is convenient, but since an `.ipynb` file is just JSON, you can also build the structure manually as a Python dictionary and write it with the `json` library. This shows exactly what the schema looks like under the hood.

In [ ]:
# Build the same notebook structure manually as a dictionary
raw_nb = {
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": ["# Raw JSON Notebook\n", "\n", "Built without nbformat."],
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": ["print('Hello from a hand-built notebook!')"],
        },
    ],
    "metadata": {
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python", "version": "3.10"},
    },
    "nbformat": 4,
    "nbformat_minor": 5,
}

raw_path = Path("raw_json_notebook.ipynb")
with raw_path.open("w", encoding="utf-8") as f:
    json.dump(raw_nb, f, indent=2)

print(f"Raw JSON notebook written to: {raw_path.resolve()}")

# Confirm nbformat also considers this hand-built file valid
nbformat.validate(nbformat.read(raw_path, as_version=4))
print("Raw JSON notebook is valid!")